In [0]:
%sql
create catalog if not exists dev;

In [0]:
#Paqueterías
%pip install geoip2
import ast
from pyspark.sql.functions import col, udf, explode, expr, array_max, array_agg, array_min, aggregate, size
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, LongType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
import pyspark.sql.functions as F
import uuid
from pyspark.sql.functions import monotonically_increasing_id
# Esto se hace dentro de una UDF (User Defined Function)
import geoip2.database

In [0]:
# Lectura (Con el separador '|')
file_path = "/Volumes/dev/ciencias_data/session_data/sessions_part2.csv"

df_raw = spark.read.csv(
    file_path,
    header=True,
    sep="|", 
    inferSchema=True,
    quote='"',
    escape='"'
)

df_raw.show()

In [0]:
# Definición de Esquema
element_schema = StructType([
    StructField("srcDataBytes", LongType(), True),
    StructField("dstBytes", LongType(), True),
    StructField("packetLen", ArrayType(LongType()), True),
    StructField("srcPort", LongType(), True),
    StructField("totPackets", LongType(), True),
    StructField("packetPos", ArrayType(LongType()), True),
    StructField("srcPayload", DoubleType(), True),
    StructField("segmentCnt", LongType(), True),
    StructField("srcPackets", LongType(), True),
    StructField("protocol", StringType(), True),
    StructField("lastPacket", LongType(), True),
    StructField("dstPort", LongType(), True),
    StructField("communityId", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("srcASN", StringType(), True),
    StructField("dstASN", StringType(), True)
])


output_schema = ArrayType(element_schema)

In [0]:
#--- G E M I N I ---#
from pyspark.sql.types import *

# Definimos el esquema interno para la columna "data"
data_internal_schema = StructType([
    # --- Identificación y Red Básica ---
    StructField("communityId", StringType(), True),
    StructField("srcIp", StringType(), True),
    StructField("dstIp", StringType(), True),
    StructField("srcPort", IntegerType(), True),
    StructField("dstPort", IntegerType(), True),
    StructField("ipProtocol", IntegerType(), True),
    StructField("protocol", ArrayType(StringType()), True),
    StructField("protocolCnt", IntegerType(), True),
    StructField("node", StringType(), True),
    
    # --- Métricas de Paquetes y Bytes ---
    StructField("totPackets", IntegerType(), True),
    StructField("srcPackets", IntegerType(), True),
    StructField("dstPackets", IntegerType(), True),
    StructField("totBytes", LongType(), True),
    StructField("srcBytes", LongType(), True),
    StructField("dstBytes", LongType(), True),
    StructField("totDataBytes", LongType(), True),
    StructField("srcDataBytes", LongType(), True),
    StructField("dstDataBytes", LongType(), True),
    StructField("packetLen", ArrayType(IntegerType()), True),
    
    # --- Timestamps (Originales en Long) ---
    StructField("timestamp", LongType(), True),
    StructField("firstPacket", LongType(), True),
    StructField("lastPacket", LongType(), True),
    
    # --- Direcciones MAC y Capa Física ---
    StructField("srcMac", ArrayType(StringType()), True),
    StructField("srcMacCnt", IntegerType(), True),
    StructField("dstMac", ArrayType(StringType()), True),
    StructField("dstMacCnt", IntegerType(), True),
    
    # --- Payloads (Carga Útil) ---
    StructField("srcPayload8", StringType(), True),
    StructField("dstPayload8", StringType(), True),
    StructField("segmentCnt", IntegerType(), True),
    
    # --- Información de DNS (Si existe en el registro) ---
    StructField("dns", StructType([
        StructField("ip", ArrayType(StringType()), True),
        StructField("host", ArrayType(StringType()), True),
        StructField("type", ArrayType(StringType()), True),
        StructField("status", ArrayType(StringType()), True),
        StructField("ASN", ArrayType(StringType()), True),
        StructField("RIR", ArrayType(StringType()), True),
        StructField("ipCnt", IntegerType(), True),
        StructField("hostCnt", IntegerType(), True)
    ]), True),
    
    # --- Otros Metadatos ---
    StructField("fileId", ArrayType(LongType()), True),
    StructField("length", IntegerType(), True),
    StructField("srcASN", StringType(), True),
    StructField("dstASN", StringType(), True),
    StructField("vlan", ArrayType(IntegerType()), True)
])

# Esquema para la lectura inicial del archivo
session_root_schema = StructType([
    StructField("data", StringType(), True),
    StructField("timestamp", StringType(), True)
])

In [0]:
# G E M I N I #
from pyspark.sql.functions import from_json, col

# 1. Leer el archivo usando el separador pipe '|'
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", "|") \
    .schema(session_root_schema) \
    .load("/Volumes/dev/ciencias_data/session_data/sessions_part1.csv")

# 2. Convertir el String JSON a columnas usando el esquema completo
df_parsed = df_raw.withColumn("data_struct", from_json(col("data"), data_internal_schema))

# 3. Seleccionar todos los campos internos y el timestamp del archivo
# Usamos data_struct.* para expandir automáticamente todas las columnas
df_final = df_parsed.select("data_struct.*", col("timestamp").alias("file_timestamp"))

# Mostrar el resultado
df_final.display()

In [0]:
# 1. Leemos el JSON como una columna de texto normal primero
# 2. Usamos get_json_object para extraer 'vlan' directamente sin esquema
df_check = df_raw.withColumn("vlan_raw", F.get_json_object(F.col("data"), "vlan"))

df_check.select("vlan_raw").distinct().display()

In [0]:
def parse_safe(data_str):
    """
    Intenta parsear la cadena. Si falla por CUALQUIER razón, 
    devuelve una lista vacía para no romper el proceso.
    """
    if data_str is None:
        return []
    
    try:
        # Intentamos evaluar la estructura
        parsed = ast.literal_eval(data_str)
        
        # Verificamos que sea una lista (es lo que espera el esquema)
        if isinstance(parsed, list):
            return parsed
        elif isinstance(parsed, dict):
            return [parsed] # Si es un solo dict, lo metemos en una lista
        return []
        
    except Exception as e:
        # Capturamos TODO (Exception). 
        # Si la cadena está cortada, tiene caracteres raros o el delimitador rompió el string,
        # simplemente devolvemos lista vacía y seguimos con la siguiente fila.
        return []

In [0]:
# Registramos la UDF segura
parse_udf_safe = udf(parse_safe, output_schema)

# Procesamiento

# Aplicamos la UDF
df_parsed = df_raw.withColumn("data_parsed", parse_udf_safe(col("data")))

# Filtramos las filas que no se pudieron leer (opcional, para limpieza)
# df_parsed = df_parsed.filter(size(col("data_parsed")) > 0)

# Explode y selección
df_exploded = df_parsed.select(explode(col("data_parsed")).alias("col"))
df_final = df_exploded.select("col.*")

df_final_with_id = df_final.withColumn("id", monotonically_increasing_id())
print("Mostrando resultados procesados:")
df_final_with_id.display()

In [0]:
'''Sustituya la columna packetLen por los sumarizados: suma total, media de los paquetes,
el mínimo, máximo valor de paquetes. La longitud de paquetes de red se refiere al tamaño
de los fragmentos o unidades de datos que se transmiten a través de una red informática
 
 #Explode sobre columnas que son arrays para obetener un registro de asistencia por fila
sumarizados=df_final.select("srcDataBytes","dstBytes","srcPort","totPackets","packetPos","srcPayload","segmentCnt","srcPackets","protocol","lastPacket","dstPort","communityId","timestamp","srcASN","dstASN",F.explode(F.col("packetLen")).alias("Sumarizados"))
sumarizados.limit(10).display()'''


#df_metricas = df_final_with_id.withColumn("Suma_Total",expr("aggregate(packetLen, 0L, (acc,x) -> acc+x)"))
#df_metricas = df_final_with_id.withColumn("Media",expr("aggregate(packetLen, 0L, (acc,x) -> acc+x) / size(packetLen)"))
#df_metricas = df_final_with_id.withColumn("Minimo_Total",array_min(F.col("packetLen")))
#df_metricas = df_final_with_id.withColumn("Maximo_Total",array_max(F.col("packetLen")))

df_Metricas=(df_final_with_id.withColumn("Suma_Total",expr("aggregate(packetLen, 0L, (acc,x) -> acc+x)"))
 .withColumn("Media",expr("aggregate(packetLen, 0L, (acc,x) -> acc+x) / size(packetLen)"))
 .withColumn("Minimo_Total",array_min(F.col("packetLen"))).withColumn("Minimo_Total",array_min(F.col("packetLen"))).withColumn("Maximo_Total",array_max(F.col("packetLen"))).drop("packetLen")
)
df_Metricas.limit(5).display()

'''
df_minimo = df_final_with_id.withColumn("Minimo_Total",expr("min(packetLen)"))

df_maximo = df_final_with_id.withColumn("Maximo_Total",expr("max(packetLen)"))
'''
#df_metricas.display()

In [0]:
# Con withColumn se agregan columnas y con un valor constante usando lit, ambas columnas agregaron valor de tiempo.
'''
df_tiempo=(df_Metricas.withColumn("timestamp",F.lit(F.current_timestamp()))
    .withColumn("unix_timestamp",F.lit(F.unix_timestamp() ))
)
df_tiempo.limit(5).display()'''

df_limpio = df_Metricas.withColumn(
    "lastPacket", 
    (F.col("lastPacket") / 1000).cast("timestamp")
)

df_limpio.display()

In [0]:
                                                            ## - - -CSV 2 - - - ##
# Lectura (Con el separador '|')
file_path = "/Volumes/dev/ciencias_data/session_data/sessions_part2.csv"

df_raw = spark.read.csv(
    file_path,
    header=True,
    sep="|", 
    inferSchema=True,
    quote='"',
    escape='"'
)

df_raw.show()

In [0]:
element_schema = StructType([
    StructField("srcDataBytes", LongType(), True),
    StructField("dstBytes", LongType(), True),
    StructField("packetLen", ArrayType(LongType()), True),
    StructField("srcPort", LongType(), True),
    StructField("totPackets", LongType(), True),
    StructField("packetPos", ArrayType(LongType()), True),
    StructField("srcPayload", StringType(), True),
    StructField("segmentCnt", LongType(), True),
    StructField("srcPackets", LongType(), True),
    StructField("protocol", StringType(), True),
    StructField("lastPacket", LongType(), True),
    StructField("dstPort", LongType(), True),
    StructField("communityId", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("srcASN", StringType(), True),
    StructField("dstASN", StringType(), True)
])


output_schema = ArrayType(element_schema)

In [0]:
def parse_safe(data_str):
    """
    Intenta parsear la cadena. Si falla por CUALQUIER razón, 
    devuelve una lista vacía para no romper el proceso.
    """
    if data_str is None:
        return []
    
    try:
        # Intentamos evaluar la estructura
        parsed = ast.literal_eval(data_str)
        
        # Verificamos que sea una lista (es lo que espera el esquema)
        if isinstance(parsed, list):
            return parsed
        elif isinstance(parsed, dict):
            return [parsed] # Si es un solo dict, lo metemos en una lista
        return []
        
    except Exception as e:
        # Capturamos TODO (Exception). 
        # Si la cadena está cortada, tiene caracteres raros o el delimitador rompió el string,
        # simplemente devolvemos lista vacía y seguimos con la siguiente fila.
        return []

In [0]:
# Registramos la UDF segura
parse_udf_safe = udf(parse_safe, output_schema)

# Procesamiento

# Aplicamos la UDF
df_parsed = df_raw.withColumn("data_parsed", parse_udf_safe(col("data")))

# Filtramos las filas que no se pudieron leer (opcional, para limpieza)
# df_parsed = df_parsed.filter(size(col("data_parsed")) > 0)

# Explode y selección
df_exploded = df_parsed.select(explode(col("data_parsed")).alias("col"))
df_final = df_exploded.select("col.*")

df_final_with_id = df_final.withColumn("id", monotonically_increasing_id())
print("Mostrando resultados procesados:")
df_final_with_id.display()